# FoSp-Temporal — train trên máy (RTX 5070 Laptop)

Notebook chạy `research/temporal_fosp/train_temporal.py` (bản build một file) như một tiến trình con và cập nhật bảng ngay trong notebook:
tiến độ, nhiệt độ GPU, **F1 / IoU / precision / recall** của từng lần kiểm tra val, rồi kết quả test, báo động giả và bootstrap theo ngày.

* E1 (ảnh đơn) và E2 (nền quan sát + phân xử) train **cùng số update**, checkpoint chọn **chỉ theo val**, test chấm **một lần** ở cuối.
* Laptop: tạm dừng khi GPU ≥ 90 °C hai lần liên tiếp (~1 phút), chạy tiếp khi ≤ 75 °C; nghỉ 10 phút sau mỗi 75 phút train.
* Mọi file tạm, cache và kết quả nằm trên ổ D (`work/fw_local`).
* Chạy nền: `work/fw_overnight.ps1` (papermill, tự lưu notebook mỗi 30 giây). Mở xem: `work/open_jupyter.ps1`.

In [ ]:
# Parameters (papermill can override these)
MODE = 'smoke+long'   # 'smoke', 'long', 'smoke+long' or 'none'
UPDATES = 0           # 0 = choose from the smoke test so that E1 + E2 fit in TRAIN_HOURS
TRAIN_HOURS = 5.5     # training time of E1 + E2 together (thermal pauses and rests excluded)
N_VAL_EVALS = 8       # validation checks per run; the best one (image-F1) is kept
RUN_NAME = ''         # output folder under work/fw_local (default long_E1E2_u<N>)
RUNS = 'E1,E2'        # E1 = single frame, E2 = observed background + reliability gate
FUSION = 'residual'   # E2 fusion: 'residual' (zero-init additive branch) or 'convex' (Day-2 pilot)
INIT_BIAS = 0.0       # arbitration prior: 0 -> gate 0.5 at start (the convex pilot used 2.5)

In [ ]:
import html, json, os, re, subprocess, sys, time
from collections import deque
from pathlib import Path

REPO = Path(os.environ.get('FW_REPO', r'D:\ForestWatch'))
HERE = REPO / 'research' / 'temporal_fosp'
ROOT = REPO / 'work' / 'fw_local'
PY = sys.executable
D = lambda *p: str(REPO.joinpath(*p))
ENV = {  # same as work/fw_local_env.ps1 — everything on drive D
    'FW_KAGGLE_ROOT': str(ROOT),
    'FW_FOSP_REPO': D('outputs', 'forestwatch-mvp', '.fosp-runtime', 'FoSp'),
    'FW_MMCV_DIR': D('outputs', 'forestwatch-mvp', '.fosp-runtime', 'mmcv'),
    'FW_DEPS_DIR': D('work', 'fw_local', 'no-extra-deps'),
    'FW_NO_PIP': '1',
    'FW_CKPT': D('checkpoints', 'fosp_smoke5k_14000', 'best.pth'),
    'FW_SEG_ROOT': D('datasets', 'SmokeSeg-data', 'SmokeSeg'),
    'FW_MANIFEST': D('datasets', 'fw_day2_prep', 'fw_day2', 'manifest_day2.json'),
    'FW_WORKERS': '0', 'FW_MICRO_BATCH': '1', 'FW_ACCUM': '6', 'FW_EVAL_BATCH': '2',
    'FW_THERMAL_MAX': '90', 'FW_THERMAL_RESUME': '75', 'FW_REST_EVERY': '4500', 'FW_REST_SECONDS': '600',
    'FW_TOTAL_BUDGET': '172800', 'PYTHONUNBUFFERED': '1',
    'TEMP': D('work', 'temp'), 'TMP': D('work', 'temp'), 'TORCH_HOME': D('work', 'cache', 'torch'),
    'XDG_CACHE_HOME': D('work', 'cache'), 'CUDA_CACHE_PATH': D('work', 'cache', 'nv-compute'),
    'MPLCONFIGDIR': D('work', 'cache', 'matplotlib'), 'PIP_CACHE_DIR': D('work', 'cache', 'pip'),
}
for k in ('TEMP', 'TORCH_HOME', 'CUDA_CACHE_PATH', 'MPLCONFIGDIR'):
    Path(ENV[k]).mkdir(parents=True, exist_ok=True)
os.environ.update({k: ENV[k] for k in ('TEMP', 'TMP', 'MPLCONFIGDIR', 'XDG_CACHE_HOME')})
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([PY, str(HERE / 'build_kaggle_script.py')], check=True)
SCRIPT = HERE / 'build' / 'train_temporal_kaggle.py'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total,temperature.gpu', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())

## Kiểm tra dữ liệu
Mọi ảnh, mask, nền và khung âm tính trong manifest phải tồn tại và mở được. File thiếu hoặc hỏng được tải lại từ output Kaggle (`work/bin/kaggle.cmd`).

In [ ]:
from PIL import Image

def bad_files():
    m = json.loads(Path(ENV['FW_MANIFEST']).read_text(encoding='utf-8'))
    prep, seg = Path(ENV['FW_MANIFEST']).parent, Path(ENV['FW_SEG_ROOT'])
    rows = m['train'] + m['val'] + m['test']
    paths = [seg / r[k] for r in rows for k in ('image', 'mask')] + [prep / r['ref'] for r in rows if r.get('ref')]
    paths += [prep / n[k] for n in m['negatives'] for k in ('image', 'ref')]
    bad = []
    for p in paths:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad.append(p)
    return paths, bad

paths, bad = bad_files()
print(f'{len(paths)} files checked, {len(bad)} missing or unreadable')
if bad:
    prep_bad = [p for p in bad if 'fw_day2_prep' in str(p)]
    assert len(prep_bad) == len(bad), f'SmokeSeg files are bad: {bad[:3]}'
    for p in prep_bad:
        p.unlink(missing_ok=True)
    names = '|'.join(re.escape(p.name) for p in prep_bad[:150])
    cmd = [D('work', 'bin', 'kaggle.cmd'), 'kernels', 'output', 'nguynlduyb24dckh043/forestwatch-day2-temporal-prep',
           '-p', D('datasets', 'fw_day2_prep'), '-q', '--page-size', '200', '-o']
    if len(prep_bad) <= 150:
        cmd += ['--file-pattern', f'({names})$']
    subprocess.run(cmd, check=False)
    paths, bad = bad_files()
    print(f'after re-download: {len(bad)} still bad')
assert not bad, bad[:5]

## Bảng theo dõi

In [ ]:
METRICS = [('image_f1', 'image-F1'), ('image_iou', 'image-IoU'), ('pixel_f1', 'pixel-F1'),
           ('precision', 'P (pixel)'), ('recall', 'R (pixel)')]
EVENT_TAGS = {'ENV', 'DATA', 'DONORS', 'RUN_CONFIG', 'INIT', 'TARGET_UPDATES', 'TRAIN_DONE', 'STOP_EARLY',
              'THERMAL_PAUSE', 'THERMAL_RESUME', 'REST', 'ALL_DONE'}

def fmt(v, nd=4):
    return '—' if v is None else (f'{v:.{nd}f}' if isinstance(v, float) else html.escape(str(v)))

def table(head, rows):
    h = ''.join(f'<th style="padding:2px 8px">{html.escape(str(c))}</th>' for c in head)
    b = ''.join('<tr>' + ''.join(f'<td style="padding:2px 8px">{c}</td>' for c in r) + '</tr>' for r in rows)
    return f'<table style="border-collapse:collapse;margin:4px 0">{"<tr>" + h + "</tr>"}{b}</table>'

def absorb(st, tag, p):
    if tag == 'TRAIN':
        st['train'][p['run']] = p
    elif tag == 'VAL':
        st['val'].append(p)
    elif tag == 'BOOTSTRAP':
        st['boot'].append(p)
    elif tag.startswith('RESULT_'):
        st['results'][tag[len('RESULT_'):]] = p
    if tag in EVENT_TAGS:
        st['events'].append(f'{tag} ' + json.dumps(p, default=str)[:400])

def last_thermal(out_dir):
    try:
        return (Path(out_dir) / 'thermal.csv').read_text(encoding='utf-8').strip().splitlines()[-1].split(',')
    except Exception:
        return None

def render(st, out_dir, started):
    parts = [f'<b>{html.escape(st["tag"])}</b> · {(time.time() - started) / 60:.1f} phút · {html.escape(str(out_dir))}']
    t = last_thermal(out_dir)
    if t and len(t) >= 7:
        parts.append(f'GPU {t[3]} °C · {t[4]} W · {t[5]} MHz · {t[6]} % (run {t[1]}, update {t[2]})')
    if st['train']:
        rows = []
        for run, p in st['train'].items():
            eta = (p['of'] - p['update']) * p['sec_per_update'] / 60
            rows.append([run, f"{p['update']}/{p['of']}", fmt(p['loss']), fmt(p.get('rel')), fmt(p['sec_per_update'], 3),
                         fmt(p['peak_gb'], 2), f'{eta:.0f}'])
        parts.append(table(['run', 'update', 'loss', 'rel loss', 's/update', 'peak GB', 'còn ~phút'], rows))
    if st['val']:
        best = {}
        for v in st['val']:
            if v['run'] not in best or v['image_f1'] > best[v['run']]['image_f1']:
                best[v['run']] = v
        rows = [[v['run'], v['update'], v['n']] + [(f'<b>{fmt(v[k])}</b>' if best[v['run']] is v else fmt(v[k])) for k, _ in METRICS]
                for v in st['val']]
        parts.append('<b>Validation</b> (đậm = checkpoint tốt nhất đến lúc này, chọn theo image-F1)'
                     + table(['run', 'update', 'n'] + [m for _, m in METRICS], rows))
    res = {k: v for k, v in st['results'].items() if not k.endswith('_neg')}
    if res:
        rows = []
        for k, p in res.items():
            a, sz, off = p['all'], p['size'], p['offset']
            rows.append([k, a.get('n')] + [fmt(a.get(m)) for m, _ in METRICS]
                        + [f"{fmt(sz[b].get('image_f1'))} ({sz[b].get('n', 0)})" for b in ('small', 'medium', 'large')]
                        + [f"{fmt(off[o].get('image_f1'))} ({off[o].get('n', 0)})" for o in ('0-10min', '10-20min', '20min+')])
        parts.append('<b>Kết quả</b> (test chỉ chấm một lần, sau khi train; cột F1 theo nhóm là image-F1 (n))'
                     + table(['tập', 'n'] + [m for _, m in METRICS]
                             + ['small', 'medium', 'large', '0–10 phút', '10–20 phút', '> 20 phút'], rows))
    neg = {k: v for k, v in st['results'].items() if k.endswith('_neg')}
    if neg:
        parts.append('<b>Báo động giả trên khung không khói thật</b>' + table(
            ['run', 'n', '≥ 1 px', '≥ 50 px', '≥ 200 px', 'diện tích TB'],
            [[k, p['n'], fmt(p['fa_any'], 3), fmt(p['fa_50'], 3), fmt(p['fa_200'], 3), fmt(p['mean_area'], 6)] for k, p in neg.items()]))
    if st['boot']:
        parts.append('<b>Bootstrap ghép cặp theo ngày</b>' + table(
            ['cặp', 'split', 'metric', 'Δ', 'CI 95%', 'số ngày', 'P(Δ ≤ 0)'],
            [[b['pair'], b['split'], b['metric'], fmt(b['delta']), f"[{fmt(b['ci95'][0])}, {fmt(b['ci95'][1])}]",
              b['n_groups'], fmt(b['p_le_0'], 3)] for b in st['boot']]))
    parts.append('<details><summary>sự kiện / thông báo</summary><pre>'
                 + html.escape('\n'.join(list(st['events'])[-15:] + ['---'] + list(st['msgs'])[-15:])) + '</pre></details>')
    return '<div style="font-family:Consolas,monospace;font-size:12px">' + '<br>'.join(parts) + '</div>'

def run_watch(tag, extra):
    from IPython.display import HTML, display
    out_dir = Path(extra['FW_OUT'])
    out_dir.mkdir(parents=True, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if not k.startswith('FW_')}
    env.update(ENV)
    env.update({k: str(v) for k, v in extra.items()})
    st = dict(tag=tag, train={}, val=[], results={}, boot=[], events=deque(maxlen=300), msgs=deque(maxlen=60))
    started, last = time.time(), 0.0
    handle = display(HTML(f'<i>{tag}: đang khởi động…</i>'), display_id=True)
    log_path = ROOT / f'{tag}.log'
    with open(log_path, 'w', encoding='utf-8') as logf:
        proc = subprocess.Popen([PY, '-u', str(SCRIPT)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                                text=True, encoding='utf-8', errors='replace', cwd=str(REPO))
        try:
            for line in proc.stdout:
                logf.write(line)
                logf.flush()
                name, _, rest = line.strip().partition(' ')
                payload = None
                if rest.startswith('{'):
                    try:
                        payload = json.loads(rest)
                    except json.JSONDecodeError:
                        pass
                if isinstance(payload, dict):
                    absorb(st, name, payload)
                elif line.strip():
                    st['msgs'].append(line.rstrip()[:300])
                if time.time() - last > 5 or name in ('VAL', 'TRAIN_DONE', 'ALL_DONE') or name.startswith('RESULT_'):
                    handle.update(HTML(render(st, out_dir, started)))
                    last = time.time()
        except KeyboardInterrupt:
            proc.terminate()
            raise
        code = proc.wait()
    handle.update(HTML(render(st, out_dir, started)))
    print(f'{tag}: exit {code}, log {log_path}')
    return code, st

## 1. Chạy thử (8 ảnh mỗi split, 50 update mỗi run)
Đo tốc độ và VRAM của E1, E2 trên laptop trước khi train dài.

In [ ]:
spu = {}
if 'smoke' in MODE:
    code, smoke = run_watch('smoke', dict(FW_LIMIT=8, FW_RUNS='E1,E2', FW_UPDATES=50, FW_N_VAL_EVALS=1, FW_OUT=ROOT / 'smoke',
                                          FW_FUSION=FUSION, FW_INIT_BIAS=INIT_BIAS))
    assert code == 0 and any(e.startswith('ALL_DONE') for e in smoke['events']), f'smoke test failed, see {ROOT / "smoke.log"}'
    spu = {r: smoke['train'][r]['sec_per_update'] for r in ('E1_single', 'E2_temporal')}
    print('giây/update:', spu, '· VRAM đỉnh (GB):', {r: smoke['train'][r]['peak_gb'] for r in spu})

## 2. Train dài E1 và E2 (cùng số update)

In [ ]:
long_run, out = None, None
if 'long' in MODE:
    assert UPDATES or spu, 'set UPDATES or run the smoke test first'
    runs = [r.strip() for r in RUNS.split(',')]
    names = {'E1': 'E1_single', 'E2': 'E2_temporal'}
    n = UPDATES or int(max(1500, min(6000, TRAIN_HOURS * 3600 / sum(spu[names[r]] for r in runs) // 100 * 100)))
    out = ROOT / (RUN_NAME or f'long_{"".join(runs)}_{FUSION}_u{n}')
    print(f'{RUNS}: mỗi run {n} update, {N_VAL_EVALS} lần kiểm tra val, E2 fusion={FUSION}, kết quả ở {out}')
    code, long_run = run_watch('long', dict(FW_RUNS=RUNS, FW_UPDATES=n, FW_N_VAL_EVALS=N_VAL_EVALS, FW_OUT=out,
                                            FW_FUSION=FUSION, FW_INIT_BIAS=INIT_BIAS))
    assert code == 0, f'long run failed (exit {code}), see {ROOT / "long.log"}'

## 3. Đường val

In [ ]:
if long_run and long_run['val']:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
    for ax, (k, label) in zip(axes, [('image_f1', 'image-F1'), ('image_iou', 'image-IoU'), ('precision', 'P (pixel)'), ('recall', 'R (pixel)')]):
        for run in ('E1_single', 'E2_temporal'):
            pts = [(v['update'], v[k]) for v in long_run['val'] if v['run'] == run]
            if pts:
                ax.plot(*zip(*pts), marker='o', label=run)
        ax.set_title(label + ' (val)')
        ax.set_xlabel('update')
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(out / 'val_curves.png', dpi=120)
    plt.show()